In [ ]:
from glob import glob
import os
import pandas as pd
import json
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import warnings ; warnings.filterwarnings('ignore')

from matplotlib import pyplot as plt
import seaborn as sns
from matplotlib import font_manager as fm

font_fname = os.environ.get('PROJECT_FONT_PATH', '')  # point at any local CJK/serif .ttf
font_family = fm.FontProperties(fname=font_fname).get_name() #폰트 설정
plt.rcParams["font.family"] = font_family  #폰트 적용






def score2dict(yt, yp, n_samples = None, binary_flag = False):
    accuracy = accuracy_score(yt, yp, sample_weight = n_samples)
    if binary_flag:
        f1 = f1_score(yt, yp, average = 'binary', sample_weight = n_samples)
        weighted_f1 = f1_score(yt, yp, average = 'weighted', sample_weight = n_samples)
    else:
        f1 = f1_score(yt, yp, average = 'macro', sample_weight = n_samples)
        weighted_f1 = f1_score(yt, yp, average = 'weighted', sample_weight = n_samples)
    return dict(accuracy = accuracy,f1 = f1,weighted_f1 = weighted_f1,)



In [ ]:

rev_dir_path = os.path.join('data','reviewer')
metas_df, recur_df = [pd.read_excel(i, sheet_name=None) for i in sorted(glob(os.path.join(rev_dir_path,'*.xlsx')))]


In [ ]:
agg_df_save_dir = os.path.join('data','reviewed_labels')
load_all = pd.read_excel([i for i in glob(os.path.join(agg_df_save_dir, '*')) if '.xlsx' in i][0])
cancer_type_df = pd.read_excel('data/cancer_type_map.xlsx')

cancer_dict_path = os.path.join('.', 'cancer_dict.json')
with open(cancer_dict_path, 'r', encoding='utf-8') as f:
    cancer_dict = json.load(f)
cancer_en_dict = {int(k) : v['name_en'] for k, v in cancer_dict.items()}

In [ ]:
missing_numb = 99
cancer_en_dict[missing_numb] = 'Missing'
cancer_type_df.iloc[:,2].fillna(' ', inplace = True)
cancer_type_df.iloc[:,3].fillna(missing_numb, inplace = True)
# cancer_type_df.loc[:,'암번호'] = cancer_type_df.loc[:,'암번호'].astype(int)
cancer_type_df['암번호'] = cancer_type_df.암번호.astype(int)

def age_group(age):
    for n, (sty, edy) in enumerate(zip(age_bins[:-1], age_bins[1:])):
        if sty <= age < edy:
            return n
    else:
        return n + 1

age_bins = [0] + list(range(40, 90, 10))
print(age_bins)

age_dict = {}
for n, (st, ed) in enumerate(zip(age_bins, age_bins[1:])):
    strs = r'~'
    if n > 0:
        strs = f'{st}{strs}'
    strs = f'{strs}{ed}'
    age_dict[n] = strs
else:
    age_dict[n+1] = f'{ed}~'
print(age_dict)



In [ ]:
merged_all = pd.merge(load_all, cancer_type_df, on = ['병원등록번호', '수술일자'], how = 'left')
merged_all['index'] = merged_all.index
merged_all['나이구간'] = merged_all['검사나이'].apply(age_group)

In [ ]:
metas_labeled_df = pd.concat([metas_df[c] for c in ['negative','uncertain','positive']]).dropna()
recur_df['negative'].loc[:99, '실제'] = 'negative'
recur_labeled_df = pd.concat([recur_df[c] for c in ['negative','uncertain','positive']]).dropna()

In [ ]:
result_df = pd.DataFrame([
    {'dataset': 'recur', **score2dict(recur_labeled_df['실제'], recur_labeled_df['prediction'])},
    {'dataset': 'metas', **score2dict(metas_labeled_df['실제'], metas_labeled_df['prediction'])},
])
result_df

In [ ]:
recur_merged_df = recur_labeled_df.merge(merged_all[['index','성별','나이구간','암번호']], on = 'index', how = 'left')
metas_merged_df = metas_labeled_df.merge(merged_all[['index','성별','나이구간','암번호']], on = 'index', how = 'left')
recur_merged_df['나이구간'] = recur_merged_df['나이구간'].apply(lambda x: age_dict[x])
recur_merged_df['암번호'] = recur_merged_df['암번호'].apply(lambda x: cancer_en_dict[x])
metas_merged_df['나이구간'] = metas_merged_df['나이구간'].apply(lambda x: age_dict[x])
metas_merged_df['암번호'] = metas_merged_df['암번호'].apply(lambda x: cancer_en_dict[x])

In [ ]:
target_class = ['negative','uncertain','positive']

In [ ]:
figdir = os.path.join('.','figure')
os.makedirs(figdir, exist_ok = True)

In [ ]:
plot_df = recur_merged_df

group_keys = ['성별','나이구간', '암번호']
group_counts = [plot_df[g].nunique() for g in group_keys]
width_ratios = group_counts

# 카테고리 순서 지정
age_group_order = [age_dict[k] for k in sorted(age_dict.keys())]
cancer_type_order = [cancer_en_dict[k] for k in cancer_en_dict.keys()]

plt.rcParams['font.size'] = 15
fig, axes = plt.subplots(
    2, 3, figsize=(15, 8),
    gridspec_kw={'width_ratios': width_ratios},
    # sharex = True
)






for n_row, col_axes in enumerate(axes):
    if n_row == 0:
        plot_df = recur_merged_df
    else:
        plot_df = metas_merged_df

    for n, (c, en_c) in enumerate(zip(['성별','나이구간','암번호'], ['Sex','Age','Cancer Type'])):
        # x_order 지정
        if c == '나이구간':
            order = age_group_order
        elif c == '암번호':
            order = cancer_type_order
        else:
            order = sorted(plot_df[c].unique())  # 성별은 오름차순

        for g, sdf in plot_df.groupby(c):
            # 카테고리 순서를 유지하기 위해 c 컬럼을 카테고리 타입으로 일시적으로 지정
            sdf_counts = sdf.groupby([c, '실제']).size().reset_index(name='n_samples')
            if c == '나이구간':
                cat_type = pd.CategoricalDtype(categories=age_group_order, ordered=True)
                sdf_counts[c] = sdf_counts[c].astype(cat_type)
            elif c == '암번호':
                cat_type = pd.CategoricalDtype(categories=cancer_type_order, ordered=True)
                sdf_counts[c] = sdf_counts[c].astype(cat_type)
            # 모든 조합에 대해 0을 채워주기 위해 MultiIndex로 재구성
            index_vals = order  # x축 (카테고리 순서)
            hue_vals = target_class  # '실제' 클래스의 모든 값
            mi = pd.MultiIndex.from_product([index_vals, hue_vals], names=[c, '실제'])
            sdf_counts_full = sdf_counts.set_index([c, '실제']).reindex(mi, fill_value=0).reset_index()

            sns.barplot(
                data=sdf_counts_full,
                x=c,
                y='n_samples',
                hue='실제',
                ax=col_axes[n],
                palette={'negative': 'tab:blue', 'uncertain': 'tab:green', 'positive': 'tab:red'},
                orient='v',
                edgecolor='black',
                order=order,
                hue_order=target_class,
            )

            # 팔레트 일치를 위해 handles를 legend에서 먼저 받아 둠
            handles, labels = col_axes[n].get_legend_handles_labels()


            col_axes[n].legend().remove()
            col_axes[n].set_xlabel('')
            col_axes[n].set_ylabel('')

            if n_row == 0:
                col_axes[n].set_title(en_c)
                col_axes[n].set_xticklabels([])
            else:
                col_axes[n].set_title('')

            if n == 0: 
                col_axes[n].set_ylabel(['Recurrence', 'Metastasis'][n_row])

    # if n_row > 0:
    for n, ax in enumerate(col_axes):
        labels = [label.get_text() for label in ax.get_xticklabels()]
        new_labels = []
        for l in labels:
            if ('&' in l) and (len(l) >= 20):
                idx = l.index('&')
                new_l = l[:idx].rstrip() + '\n&' + l[idx+1:]
                new_labels.append(new_l)
            else:
                new_labels.append(l)
        ax.set_xticklabels(new_labels, rotation=45 if n == 2 else 0)
    # else:
        
    # handles의 색상으로 legend 그리기
    if n_row == 0:  # 한 번만 legend 생성(혹은 원하는 위치에서)
        # Macro-F1, Weighted F1, Accuracy 순서로 맞춰서 정렬
        handles_dict = {label.lower(): h for h, label in zip(handles, labels)}
        legend_labels = ['Negative','Uncertain','Positive']
        legend_handles = [handles_dict.get(lbl.lower(), h) for lbl, h in zip(legend_labels, handles)]
        fig.legend(
            handles=legend_handles,
            labels=legend_labels,
            loc='lower left',
            bbox_to_anchor=(0.05, 0.025),
            # fontsize=12,
            title='Class',
            ncol=3
        )


fig.tight_layout()
fig.savefig(os.path.join(figdir, 'rev3_category_histogram.png'), dpi = 400)
        # axes[n].set_xscale('log')

In [ ]:
plot_df = recur_merged_df

group_keys = ['성별','나이구간', '암번호']
group_counts = [plot_df[g].nunique() for g in group_keys]
width_ratios = group_counts

# 카테고리 순서 지정
age_group_order = [age_dict[k] for k in sorted(age_dict.keys())]
cancer_type_order = [cancer_en_dict[k] for k in cancer_en_dict.keys()]

plt.rcParams['font.size'] = 15
fig, axes = plt.subplots(
    2, 3, figsize=(15, 8),
    gridspec_kw={'width_ratios': width_ratios},
    sharey = True
    # sharex = True
)





for n_row, col_axes in enumerate(axes):
    if n_row == 0:
        plot_df = recur_merged_df
    else:
        plot_df = metas_merged_df


    
    for n, (ax, c, en_c) in enumerate(zip(col_axes, ['성별','나이구간','암번호'], ['Sex','Age','Cancer Type'])):
        # x_order 지정
        if c == '나이구간':
            order = age_group_order
        elif c == '암번호':
            order = cancer_type_order
        else:
            order = sorted(plot_df[c].unique())  # 성별은 오름차순

        rows = []
        for g, sdf in plot_df.groupby(c):
                        
            for cls, cls_sdf in sdf.groupby('실제'):
                score_dict = score2dict(cls_sdf['실제'], cls_sdf['prediction'])
                for k, v in score_dict.items():
                    rows.append({c:g, '실제': cls, 'metric': k, 'value': v})
                
        sdf_counts = pd.DataFrame(rows)

        if c == '나이구간':
            cat_type = pd.CategoricalDtype(categories=age_group_order, ordered=True)
            sdf_counts[c] = sdf_counts[c].astype(cat_type)
        elif c == '암번호':
            cat_type = pd.CategoricalDtype(categories=cancer_type_order, ordered=True)
            sdf_counts[c] = sdf_counts[c].astype(cat_type)

        # display(sdf_counts)
        sns.barplot(
            data=sdf_counts,
            x=c,
            y='value',
            hue='metric',
            ax=ax,
            palette='husl',
            orient='v',
            edgecolor='black',
            # ci=None, # 표준편차(신뢰구간) 막대기를 그리지 않음, 
            hue_order = ['accuracy','weighted_f1','f1']
        )

        # 팔레트 일치를 위해 handles를 legend에서 먼저 받아 둠
        handles, labels = ax.get_legend_handles_labels()

        ax.legend().remove()
        ax.set_xlabel('')
        ax.set_ylabel('')

        if n_row == 0:
            col_axes[n].set_title(en_c)
            col_axes[n].set_xticklabels([])
        else:
            col_axes[n].set_title('')

        if n == 0: 
            col_axes[n].set_ylabel(['Recurrence', 'Metastasis'][n_row])

    # if n_row > 0:
    for n, ax in enumerate(col_axes):
        labels_xtick = [label.get_text() for label in ax.get_xticklabels()]
        new_labels = []
        for l in labels_xtick:
            if ('&' in l) and (len(l) >= 20):
                idx = l.index('&')
                new_l = l[:idx].rstrip() + '\n&' + l[idx+1:]
                new_labels.append(new_l)
            else:
                new_labels.append(l)
        ax.set_xticklabels(new_labels, rotation=45 if n == 2 else 0)

        # # 팔레트 일치를 위해 handles를 legend에서 먼저 받아 둠
        # handles, labels = ax.get_legend_handles_labels()

    # handles의 색상으로 legend 그리기
    if n_row == 0:  # 한 번만 legend 생성(혹은 원하는 위치에서)
        # Macro-F1, Weighted F1, Accuracy 순서로 맞춰서 정렬
        handles_dict = {label.lower(): h for h, label in zip(handles, labels)}
        legend_labels = ['Accuracy', 'Weighted F1', 'Macro-F1']
        legend_handles = [handles_dict.get(lbl.lower(), h) for lbl, h in zip(legend_labels, handles)]
        fig.legend(
            handles=legend_handles,
            labels=legend_labels,
            loc='lower left',
            bbox_to_anchor=(0.05, 0.025),
            # fontsize=12,
            title='Metric',
            ncol=3
        )

fig.suptitle('')  # 혹시 fig.suptitle을 이미 썼다면 무시됨
fig.tight_layout()
fig.savefig(os.path.join(figdir, 'rev3_category_metric_vis_class-represent-std.png'), dpi = 400)

In [ ]:
plot_df = recur_merged_df

group_keys = ['성별','나이구간', '암번호']
group_counts = [plot_df[g].nunique() for g in group_keys]
width_ratios = group_counts

# 카테고리 순서 지정
age_group_order = [age_dict[k] for k in sorted(age_dict.keys())]
cancer_type_order = [cancer_en_dict[k] for k in cancer_en_dict.keys()]

plt.rcParams['font.size'] = 15
fig, axes = plt.subplots(
    2, 3, figsize=(15, 8),
    gridspec_kw={'width_ratios': width_ratios},
    sharey = True
    # sharex = True
)





for n_row, col_axes in enumerate(axes):
    if n_row == 0:
        plot_df = recur_merged_df
    else:
        plot_df = metas_merged_df


    
    for n, (ax, c, en_c) in enumerate(zip(col_axes, ['성별','나이구간','암번호'], ['Sex','Age','Cancer Type'])):
        # x_order 지정
        if c == '나이구간':
            order = age_group_order
        elif c == '암번호':
            order = cancer_type_order
        else:
            order = sorted(plot_df[c].unique())  # 성별은 오름차순

        rows = []
        for g, sdf in plot_df.groupby(c):
                        
            for cls, cls_sdf in sdf.groupby('실제'):
                score_dict = score2dict(cls_sdf['실제'], cls_sdf['prediction'])
                for k, v in score_dict.items():
                    rows.append({c:g, '실제': cls, 'metric': k, 'value': v})
                
        sdf_counts = pd.DataFrame(rows).query('metric == "accuracy"')

        if c == '나이구간':
            cat_type = pd.CategoricalDtype(categories=age_group_order, ordered=True)
            sdf_counts[c] = sdf_counts[c].astype(cat_type)
        elif c == '암번호':
            cat_type = pd.CategoricalDtype(categories=cancer_type_order, ordered=True)
            sdf_counts[c] = sdf_counts[c].astype(cat_type)

        # display(sdf_counts)
        sns.barplot(
            data=sdf_counts,
            x=c,
            y='value',
            hue='실제',
            ax=ax,
            palette={'negative': 'tab:blue', 'uncertain': 'tab:green', 'positive': 'tab:red'},
            orient='v',
            edgecolor='black',
            # ci=None, # 표준편차(신뢰구간) 막대기를 그리지 않음, 
            hue_order = ['negative','uncertain','positive']
        )

        # 팔레트 일치를 위해 handles를 legend에서 먼저 받아 둠
        handles, labels = ax.get_legend_handles_labels()

        ax.legend().remove()
        ax.set_xlabel('')
        ax.set_ylabel('')

        if n_row == 0:
            col_axes[n].set_title(en_c)
            col_axes[n].set_xticklabels([])
        else:
            col_axes[n].set_title('')

        if n == 0: 
            col_axes[n].set_ylabel(['Recurrence', 'Metastasis'][n_row])

    # if n_row > 0:
    for n, ax in enumerate(col_axes):
        labels_xtick = [label.get_text() for label in ax.get_xticklabels()]
        new_labels = []
        for l in labels_xtick:
            if ('&' in l) and (len(l) >= 20):
                idx = l.index('&')
                new_l = l[:idx].rstrip() + '\n&' + l[idx+1:]
                new_labels.append(new_l)
            else:
                new_labels.append(l)
        ax.set_xticklabels(new_labels, rotation=45 if n == 2 else 0)

    # handles의 색상으로 legend 그리기
    if n_row == 0:  # 한 번만 legend 생성(혹은 원하는 위치에서)
        # Macro-F1, Weighted F1, Accuracy 순서로 맞춰서 정렬
        handles_dict = {label.lower(): h for h, label in zip(handles, labels)}
        legend_labels = ['Negative', 'Uncertain', 'Positive']
        legend_handles = [handles_dict.get(lbl.lower(), h) for lbl, h in zip(legend_labels, handles)]
        fig.legend(
            handles=legend_handles,
            labels=legend_labels,
            loc='lower left',
            bbox_to_anchor=(0.05, 0.025),
            # fontsize=12,
            title='Class',
            ncol=3
        )

        # # 팔레트 일치를 위해 handles를 legend에서 먼저 받아 둠
        # handles, labels = ax.get_legend_handles_labels()


fig.suptitle('')  # 혹시 fig.suptitle을 이미 썼다면 무시됨
fig.tight_layout()
fig.savefig(os.path.join(figdir, 'rev3_category_accuracy_via_class.png'), dpi = 400)

In [ ]:
import copy

recur_merged_df = recur_labeled_df.merge(merged_all[['index','성별','나이구간','암번호']], on = 'index', how = 'left')
metas_merged_df = metas_labeled_df.merge(merged_all[['index','성별','나이구간','암번호']], on = 'index', how = 'left')
recur_merged_df['나이구간'] = recur_merged_df['나이구간'].apply(lambda x: age_dict[x])
recur_merged_df['암번호'] = recur_merged_df['암번호'].apply(lambda x: cancer_en_dict[x])
metas_merged_df['나이구간'] = metas_merged_df['나이구간'].apply(lambda x: age_dict[x])
metas_merged_df['암번호'] = metas_merged_df['암번호'].apply(lambda x: cancer_en_dict[x])



plot_df = recur_merged_df

group_keys = ['성별','나이구간', '암번호']

cancer_merge_keys = ['Neurologic', 'Hematologic', 'Head & Neck', 'Endocrine']
merged_cancer_label = 'Neurologic & Hematologic & HeadNeck & Endocrine'

# 기존 cancer_en_dict의 키 순서 보존
cancer_keys_in_order = list(cancer_en_dict.values())

# 마지막 6개 키 추출
last6_keys = cancer_keys_in_order[-6:]

# 마지막 6개 중 merge 대상(4개)과 나머지(2개) 구분
merge_keys_found = []
non_merge_keys_found = []
for k in last6_keys:
    if k in cancer_merge_keys:
        merge_keys_found.append(k)
    else:
        non_merge_keys_found.append(k)

# 앞에 merge 4개 묶고, merged_cancer_label 이름으로 대체, 그 뒤 2개 그대로
final_keys = cancer_keys_in_order[:-6] + [merged_cancer_label] + [k for k in non_merge_keys_found]

# missing 계산 (기존과 동일)
used_cancer_names = set()
for df0 in [recur_merged_df, metas_merged_df]:
    used_cancer_names.update(df0['암번호'].map(lambda x: cancer_en_dict.get(str(x), str(x))).unique())

missing_names = []
for k, v in cancer_en_dict.items():
    if (v not in cancer_merge_keys) and (v not in [k for x in non_merge_keys_found]) and (v not in used_cancer_names):
        missing_names.append(v)

# missing 있으면 추가
if missing_names:
    final_keys += missing_names

cancer_type_order = final_keys


# plot_df의 암번호 컬럼 값 변경(머지 대상 → merged_label)
def apply_merged_cancer(x):
    en = cancer_en_dict.get(str(x), x)
    if en in cancer_merge_keys:
        return merged_cancer_label
    return en

# plot_df의 암번호값(영문으로) 변환
for df0 in [recur_merged_df, metas_merged_df]:  # recur, metas 모두
    df0['암번호_머지'] = df0['암번호'].apply(apply_merged_cancer)

group_counts = [
    plot_df['성별'].nunique(),
    plot_df['나이구간'].nunique(),
    len(cancer_type_order)
]
width_ratios = group_counts

# 카테고리 순서 지정
age_group_order = [age_dict[k] for k in sorted(age_dict.keys())]

plt.rcParams['font.size'] = 15
fig, axes = plt.subplots(
    2, 3, figsize=(15, 8),
    gridspec_kw={'width_ratios': width_ratios},
    # sharex = True
)

for n_row, col_axes in enumerate(axes):
    if n_row == 0:
        plot_df = recur_merged_df
    else:
        plot_df = metas_merged_df

    for n, (c, en_c) in enumerate(zip(['성별','나이구간','암번호_머지'], ['Sex','Age','Cancer Type'])):
        # x_order 지정
        if c == '나이구간':
            order = age_group_order
        elif c == '암번호_머지':
            order = cancer_type_order
        else:
            order = sorted(plot_df[c].unique())  # 성별은 오름차순

        for g, sdf in plot_df.groupby(c):
            # 카테고리 순서를 유지하기 위해 c 컬럼을 카테고리 타입으로 일시적으로 지정
            sdf_counts = sdf.groupby([c, '실제']).size().reset_index(name='n_samples')
            if c == '나이구간':
                cat_type = pd.CategoricalDtype(categories=age_group_order, ordered=True)
                sdf_counts[c] = sdf_counts[c].astype(cat_type)
            elif c == '암번호_머지':
                cat_type = pd.CategoricalDtype(categories=cancer_type_order, ordered=True)
                sdf_counts[c] = sdf_counts[c].astype(cat_type)
            # 모든 조합에 대해 0을 채워주기 위해 MultiIndex로 재구성
            index_vals = order  # x축 (카테고리 순서)
            hue_vals = target_class  # '실제' 클래스의 모든 값
            mi = pd.MultiIndex.from_product([index_vals, hue_vals], names=[c, '실제'])
            sdf_counts_full = sdf_counts.set_index([c, '실제']).reindex(mi, fill_value=0).reset_index()

            sns.barplot(
                data=sdf_counts_full,
                x=c,
                y='n_samples',
                hue='실제',
                ax=col_axes[n],
                palette={'negative': 'tab:blue', 'uncertain': 'tab:green', 'positive': 'tab:red'},
                orient='v',
                edgecolor='black',
                order=order,
                hue_order=target_class,
            )

            # 팔레트 일치를 위해 handles를 legend에서 먼저 받아 둠
            handles, labels = col_axes[n].get_legend_handles_labels()


            col_axes[n].legend().remove()
            col_axes[n].set_xlabel('')
            col_axes[n].set_ylabel('')

            if n_row == 0:
                col_axes[n].set_title(en_c)
                col_axes[n].set_xticklabels([])
            else:
                col_axes[n].set_title('')

            if n == 0: 
                col_axes[n].set_ylabel(['Recurrence', 'Metastasis'][n_row])

    # if n_row > 0:
    for n, ax in enumerate(col_axes):
        labels = [label.get_text() for label in ax.get_xticklabels()]
        new_labels = []
        for l in labels:
            # '&'가 있으면서 길이가 긴 경우 줄바꿈
            # 새로 병합된 뭉탱이 항목도 자연스럽게 줄바꿈 추가
            if ('&' in l) and (len(l) >= 20):
                # 모든 &마다 줄바꿈이 들어가도록 변환
                import re
                new_l = re.sub(r'\s*&\s*', r'\n& ', l)
                new_labels.append(new_l)
            elif 'Neuro/Hematologic/H&N/Endocrine' in l:
                # 지저분하게 길면 보기 예쁘게 줄바꿈
                new_labels.append('Neuro/\nHemato/\nH&N/\nEndocrine')
            else:
                new_labels.append(l)
        ax.set_xticklabels(new_labels, rotation=45 if n == 2 else 0)
    # else:
        

    # handles의 색상으로 legend 그리기
    if n_row == 0:  # 한 번만 legend 생성(혹은 원하는 위치에서)
        # Macro-F1, Weighted F1, Accuracy 순서로 맞춰서 정렬
        handles_dict = {label.lower(): h for h, label in zip(handles, labels)}
        legend_labels = ['Negative', 'Uncertain', 'Positive']
        legend_handles = [handles_dict.get(lbl.lower(), h) for lbl, h in zip(legend_labels, handles)]
        fig.legend(
            handles=legend_handles,
            labels=legend_labels,
            loc='lower left',
            bbox_to_anchor=(0.07, 0.05),
            # fontsize=12,
            title='Class',
            ncol=3
        )


fig.tight_layout()
fig.savefig(os.path.join(figdir, 'rev3_category_bunch_histogram.png'), dpi = 400)
        # axes[n].set_xscale('log')

In [ ]:
cancer_merge_keys = ['Neurologic', 'Hematologic', 'Head & Neck', 'Endocrine']
merged_cancer_label = 'Neurologic & Hematologic & HeadNeck & Endocrine'
cancer_type_order = [cancer_en_dict[k] for k in cancer_en_dict.keys() if cancer_en_dict[k] not in cancer_merge_keys]
cancer_type_order.insert(-2, merged_cancer_label)


plot_df = recur_merged_df

group_keys = ['성별','나이구간', '암번호']
group_counts = [plot_df[g].nunique() for g in group_keys]
width_ratios = group_counts

# 카테고리 순서 지정
age_group_order = [age_dict[k] for k in sorted(age_dict.keys())]
# cancer_type_order = [cancer_en_dict[k] for k in cancer_en_dict.keys()]

plt.rcParams['font.size'] = 15
fig, axes = plt.subplots(
    2, 3, figsize=(15, 8),
    gridspec_kw={'width_ratios': width_ratios},
    sharey = True
)


for n_row, col_axes in enumerate(axes):
    if n_row == 0:
        plot_df = recur_merged_df
    else:
        plot_df = metas_merged_df


    
    for n, (ax, c, en_c) in enumerate(zip(col_axes, ['성별','나이구간','암번호_머지'], ['Sex','Age','Cancer Type'])):
        # x_order 지정
        if c == '나이구간':
            order = age_group_order
        elif c == '암번호_머지':
            order = cancer_type_order
        else:
            order = sorted(plot_df[c].unique())  # 성별은 오름차순

        rows = []
        for g, sdf in plot_df.groupby(c):
                        
            for cls, cls_sdf in sdf.groupby('실제'):
                score_dict = score2dict(cls_sdf['실제'], cls_sdf['prediction'])
                for k, v in score_dict.items():
                    rows.append({c:g, '실제': cls, 'metric': k, 'value': v})
                
        sdf_counts = pd.DataFrame(rows).query('metric == "accuracy"')

        cat_type = pd.CategoricalDtype(categories=order, ordered=True)
        sdf_counts[c] = sdf_counts[c].astype(cat_type)
        # if c == '나이구간':
        #     cat_type = pd.CategoricalDtype(categories=order, ordered=True)
        #     sdf_counts[c] = sdf_counts[c].astype(cat_type)
        # elif c == '암번호':
        #     cat_type = pd.CategoricalDtype(categories=order, ordered=True)
        #     sdf_counts[c] = sdf_counts[c].astype(cat_type)

        # display(sdf_counts)
        sns.barplot(
            data=sdf_counts,
            x=c,
            y='value',
            hue='실제',
            ax=ax,
            palette={'negative': 'tab:blue', 'uncertain': 'tab:green', 'positive': 'tab:red'},
            orient='v',
            edgecolor='black',
            # ci=None, # 표준편차(신뢰구간) 막대기를 그리지 않음, 
            hue_order = ['negative','uncertain','positive']
        )

        # 팔레트 일치를 위해 handles를 legend에서 먼저 받아 둠
        handles, labels = ax.get_legend_handles_labels()

        ax.legend().remove()
        ax.set_xlabel('')
        ax.set_ylabel('')

        if n_row == 0:
            col_axes[n].set_title(en_c)
            col_axes[n].set_xticklabels([])
        else:
            col_axes[n].set_title('')

        if n == 0: 
            col_axes[n].set_ylabel(['Recurrence', 'Metastasis'][n_row])

    # if n_row > 0:
    for n, ax in enumerate(col_axes):
        labels = [label.get_text() for label in ax.get_xticklabels()]
        new_labels = []
        for l in labels:
            # '&'가 있으면서 길이가 긴 경우 줄바꿈
            # 새로 병합된 뭉탱이 항목도 자연스럽게 줄바꿈 추가
            if ('&' in l) and (len(l) >= 20):
                # 모든 &마다 줄바꿈이 들어가도록 변환
                import re
                new_l = re.sub(r'\s*&\s*', r'\n&', l)
                new_labels.append(new_l)
            elif 'Neuro/Hematologic/H&N/Endocrine' in l:
                # 지저분하게 길면 보기 예쁘게 줄바꿈
                new_labels.append('Neuro/\nHemato/\nH&N/\nEndocrine')
            else:
                new_labels.append(l)
        ax.set_xticklabels(new_labels, rotation=45 if n == 2 else 0)
    # else:


    # handles의 색상으로 legend 그리기
    if n_row == 0:  # 한 번만 legend 생성(혹은 원하는 위치에서)
        # Macro-F1, Weighted F1, Accuracy 순서로 맞춰서 정렬
        handles_dict = {label.lower(): h for h, label in zip(handles, labels)}
        legend_labels = ['Negative', 'Uncertain', 'Positive']
        legend_handles = [handles_dict.get(lbl.lower(), h) for lbl, h in zip(legend_labels, handles)]
        fig.legend(
            handles=legend_handles,
            labels=legend_labels,
            loc='lower left',
            bbox_to_anchor=(0.06, 0.05),
            # fontsize=12,
            title='Class',
            ncol=3
        )


    # # handles의 색상으로 legend 그리기
    # if n_row == 0:  # 한 번만 legend 생성(혹은 원하는 위치에서)
    #     # Macro-F1, Weighted F1, Accuracy 순서로 맞춰서 정렬
    #     handles_dict = {label.lower(): h for h, label in zip(handles, labels)}
    #     legend_labels = ['Negative', 'Uncertain', 'Positive']
    #     legend_handles = [handles_dict.get(lbl.lower(), h) for lbl, h in zip(legend_labels, handles)]
    #     fig.legend(
    #         handles=legend_handles,
    #         labels=legend_labels,
    #         loc='lower left',
    #         bbox_to_anchor=(0.05, 0.025),
    #         # fontsize=12,
    #         title='Metric',
    #         ncol=3
    #     )

fig.suptitle('')  # 혹시 fig.suptitle을 이미 썼다면 무시됨
fig.tight_layout()
fig.savefig(os.path.join(figdir, 'rev3_category_bunch_class_metric.png'), dpi = 400)

In [ ]:
from copy import deepcopy
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix_from_df(
    df, 
    target_cancer = 'Merged',
    target_class = 'negative',
    target_col = '암번호_머지',
    y_true_col='실제', 
    y_pred_col='prediction',
    labels=['negative', 'uncertain', 'positive'],
    title='Confusion Matrix', 
    cmap='Blues', 
    figsize=(5,4), 
    annot=True, 
    fmt='d'
):
    """
    주어진 DataFrame에서 실제/예측값으로 confusion matrix 그려주는 함수

    Args:
        df: DataFrame
        y_true_col: 실제값 컬럼명
        y_pred_col: 예측값 컬럼명
        labels: 사용할 클래스 리스트(정렬순서)
        title: 그래프 타이틀
        cmap: colormap
        figsize: figure 크기
        annot: 값 표시 여부
        fmt: 표시 형식
    """

    
    str_target_cancer = target_cancer
    if 'Merged' == target_cancer:
        target_cancer = merged_cancer_label
    df = df.query(f'{target_col} == "{target_cancer}" & 실제 == "{target_class}"')


    y_true = df[y_true_col]
    y_pred = df[y_pred_col]

    if labels is None:
        labels = sorted(list(set(y_true) | set(y_pred)))
    
    clabels = [i.capitalize() for i in labels]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    title = rf'{str_target_cancer} - {target_class.capitalize()}'
    plt.rcParams['font.size'] = 15
    plt.figure(figsize=figsize)
    sns.heatmap(cm, annot=annot, fmt=fmt, cmap=cmap, cbar = False,
                xticklabels=clabels, yticklabels=clabels)
    plt.xlabel('Prediction')
    plt.ylabel('Golden state')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(os.path.join(figdir, f'{title}.png'), dpi = 400)
    # plt.show()

plot_confusion_matrix_from_df(recur_merged_df, 'Merged', 'negative')
plot_confusion_matrix_from_df(metas_merged_df, 'Breast', 'positive')
plot_confusion_matrix_from_df(metas_merged_df, 'Gynecologic', 'positive')
plot_confusion_matrix_from_df(metas_merged_df, 'Missing', 'uncertain')
plot_confusion_matrix_from_df(metas_merged_df, '80~', 'uncertain', '나이구간')

In [ ]:


class_dict = {'negative': 'tab:blue', 'positive': 'tab:red'}

cancer_merge_keys = ['Neurologic', 'Hematologic', 'Head & Neck', 'Endocrine']
merged_cancer_label = 'Neurologic & Hematologic & HeadNeck & Endocrine'
cancer_type_order = [cancer_en_dict[k] for k in cancer_en_dict.keys() if cancer_en_dict[k] not in cancer_merge_keys]
cancer_type_order.insert(-2, merged_cancer_label)


# plot_df = recur_merged_df

group_keys = ['성별','나이구간', '암번호']
group_counts = [plot_df[g].nunique() for g in group_keys]
width_ratios = group_counts

# 카테고리 순서 지정
age_group_order = [age_dict[k] for k in sorted(age_dict.keys())]
# cancer_type_order = [cancer_en_dict[k] for k in cancer_en_dict.keys()]

plt.rcParams['font.size'] = 15
fig, axes = plt.subplots(
    2, 3, figsize=(15, 8),
    gridspec_kw={'width_ratios': width_ratios},
    sharey = True
)



sdict = dict()
for n_row, col_axes in enumerate(axes):
    if n_row == 0:
        plot_df = recur_merged_df.copy()
        sdict['recur'] = dict()
    else:
        plot_df = metas_merged_df.copy()
        sdict['metas'] = dict()
    plot_df['실제'] = plot_df['실제'].apply(lambda x: 1 if x == 'positive' else 0)
    plot_df['prediction'] = plot_df['prediction'].apply(lambda x: 1 if x == 'positive' else 0)

    
    for n, (ax, c, en_c) in enumerate(zip(col_axes, ['성별','나이구간','암번호_머지'], ['Sex','Age','Cancer Type'])):
        # x_order 지정
        if c == '나이구간':
            order = age_group_order
        elif c == '암번호_머지':
            order = cancer_type_order
        else:
            order = sorted(plot_df[c].unique())  # 성별은 오름차순

        rows = []
        for g, sdf in plot_df.groupby(c):
                        
            for cls, cls_sdf in sdf.groupby('실제'):
                score_dict = score2dict(cls_sdf['실제'], cls_sdf['prediction'], binary_flag = True)
                for k, v in score_dict.items():
                    rows.append({c:g, '실제': cls, 'metric': k, 'value': v})
                
        sdf_counts = pd.DataFrame(rows)
        sdf_counts = sdf_counts.loc[sdf_counts['metric'].apply(lambda x: x in ['f1','accuracy']) & (sdf_counts['실제'] == 1)]

        cat_type = pd.CategoricalDtype(categories=order, ordered=True)
        sdf_counts[c] = sdf_counts[c].astype(cat_type)
        sdict['recur' if n_row == 0 else 'metas'][c] = sdf_counts
        # if c == '나이구간':
        #     cat_type = pd.CategoricalDtype(categories=order, ordered=True)
        #     sdf_counts[c] = sdf_counts[c].astype(cat_type)
        # elif c == '암번호':
        #     cat_type = pd.CategoricalDtype(categories=order, ordered=True)
        #     sdf_counts[c] = sdf_counts[c].astype(cat_type)

        # display(sdf_counts)


        colors = sns.color_palette('husl', n_colors=8)

        # colors에서 color1과 color6(0-indexed로 0번, 5번)을 임시 리스트에 담는다
        selected_colors = [colors[0], colors[5]]

        sns.barplot(
            data=sdf_counts,
            x=c,
            y='value',
            hue='metric',
            ax=ax,
            palette=selected_colors,
            orient='v',
            edgecolor='black',
            hue_order=['accuracy', 'f1']
        )


        # 팔레트 일치를 위해 handles를 legend에서 먼저 받아 둠
        handles, labels = ax.get_legend_handles_labels()

        ax.legend().remove()
        ax.set_xlabel('')
        ax.set_ylabel('')

        if n_row == 0:
            col_axes[n].set_title(en_c)
            col_axes[n].set_xticklabels([])
        else:
            col_axes[n].set_title('')

        if n == 0: 
            col_axes[n].set_ylabel(['Recurrence', 'Metastasis'][n_row])

    # if n_row > 0:
    for n, ax in enumerate(col_axes):
        labels = [label.get_text() for label in ax.get_xticklabels()]
        new_labels = []
        for l in labels:
            # '&'가 있으면서 길이가 긴 경우 줄바꿈
            # 새로 병합된 뭉탱이 항목도 자연스럽게 줄바꿈 추가
            if ('&' in l) and (len(l) >= 20):
                # 모든 &마다 줄바꿈이 들어가도록 변환
                import re
                new_l = re.sub(r'\s*&\s*', r'\n&', l)
                new_labels.append(new_l)
            elif 'Neuro/Hematologic/H&N/Endocrine' in l:
                # 지저분하게 길면 보기 예쁘게 줄바꿈
                new_labels.append('Neuro/\nHemato/\nH&N/\nEndocrine')
            else:
                new_labels.append(l)
        ax.set_xticklabels(new_labels, rotation=45 if n == 2 else 0)





    # handles의 색상으로 legend 그리기
    if n_row == 0:  # 한 번만 legend 생성(혹은 원하는 위치에서)
        # Macro-F1, Weighted F1, Accuracy 순서로 맞춰서 정렬
        handles_dict = {label.lower(): h for h, label in zip(handles, labels)}
        legend_labels = ['Accuracy',  'F1']
        legend_handles = [handles_dict.get(lbl.lower(), h) for lbl, h in zip(legend_labels, handles)]
        fig.legend(
            handles=legend_handles,
            labels=legend_labels,
            loc='lower left',
            bbox_to_anchor=(0.13, 0.05),
            # fontsize=12,
            title='Metric',
            ncol=3
        )



fig.suptitle('')  # 혹시 fig.suptitle을 이미 썼다면 무시됨
fig.tight_layout()
fig.savefig(os.path.join(figdir, 'rev3_category_bunch_binary_metric.png'), dpi = 400)

In [ ]:
# 성별, 나이구간, 암번호_머지
c = '암번호_머지'
# t = 'recur'
pd.concat([
    sdict['recur'][c].pivot(index=[c, '실제'], columns='metric', values='value'),
    sdict['metas'][c].pivot(index=[c, '실제'], columns='metric', values='value'),
]
    # how = 'inner
    # '
)


In [ ]:
# reviewer round1
# metastasis_positive
revision_01_path_dir = sorted(glob(os.path.join('data','reviewer','round1', '*positive.xlsx')))
print(revision_01_path_dir)
rev_01_meta_path, rev_01_recur_path = revision_01_path_dir
revision_dict = dict()
revision_dict[1] = dict(
    metas = pd.read_excel(rev_01_meta_path, engine = 'openpyxl').dropna(),
    recur = pd.read_excel(rev_01_recur_path, engine = 'openpyxl').dropna()
)

revision_02_dir = os.path.join('data','reviewer','round2')
revision_02_meta_path_dir = sorted(glob(os.path.join(revision_02_dir, '*meta*.xlsx')))
revision_02_recur_path_dir = sorted(glob(os.path.join(revision_02_dir, '*recur*.xlsx')))

print(revision_02_meta_path_dir)
print(revision_02_recur_path_dir)
revision_dict[2] = dict(
    metas = pd.concat([pd.read_excel(p, engine = 'openpyxl').dropna() for p in revision_02_meta_path_dir]),
    recur = pd.concat([pd.read_excel(p, engine = 'openpyxl').dropna() for p in revision_02_recur_path_dir])
)

In [ ]:
dup_index = revision_dict[1]['metas']['index'].isin(revision_dict[2]['metas']['index'])

In [ ]:
front_df = pd.DataFrame(columns = ['Target','Predictor','Score'])
tab_df = pd.DataFrame()
for target in ['recur', 'metas']:
    sdf1 = revision_dict[1][target].loc[revision_dict[1][target]['index'].isin(revision_dict[2][target]['index'])].sort_values('index')
    sdf2 = revision_dict[2][target].loc[revision_dict[2][target]['index'].isin(revision_dict[1][target]['index'])].sort_values('index')
    human_acc = (sdf1.label.values == sdf2.재분류.values).mean()
    front_df = front_df._append(pd.DataFrame({'Target': target, 'Predictor': 'Human', 'Score': human_acc}, index = [0]))
    tab_df.loc[target, 'Human'] = human_acc
    # print(human_acc)

    human_binary_acc = (sdf1.label.apply(lambda x: 1 if x == 'positive' else 0).values == sdf2.재분류.apply(lambda x: 1 if x == 'positive' else 0).values).mean()
    front_df = front_df._append(pd.DataFrame({'Target': target, 'Predictor': 'Human', 'Score': human_binary_acc}, index = [0]))
    tab_df.loc[target, 'Human-binary'] = human_binary_acc
    # print(human_binary_acc)

    if target == 'recur':
        plot_df = recur_merged_df.copy()
    else:
        plot_df = metas_merged_df.copy()

    LM_acc = (plot_df['실제'] == plot_df['prediction']).mean()
    front_df = front_df._append(pd.DataFrame({'Target': target, 'Predictor': 'LM', 'Score': LM_acc}, index = [0]))
    tab_df.loc[target, 'LM'] = LM_acc

    
    LM_binary_acc = (plot_df['실제'].apply(lambda x: 1 if x == 'positive' else 0) == plot_df['prediction'].apply(lambda x: 1 if x == 'positive' else 0)).mean()
    front_df = front_df._append(pd.DataFrame({'Target': target, 'Predictor': 'LM-binary', 'Score': LM_binary_acc}, index = [0]))
    tab_df.loc[target, 'LM-binary'] = LM_binary_acc
    # rule_acc = (plot_df['실제'] == plot_df['rule_label']).mean()
    # front_df = front_df._append(pd.DataFrame({'Target': target, 'Predictor': 'Rule', 'Score': rule_acc}, index = [0]))

front_df.reset_index(drop = True, inplace = True)
    # front_df.loc[target] = [human_acc, LM_acc, rule_acc]
    # front_df.loc[target, 'LM'] = LM_acc
    # front_df.loc[target, 'Rule'] = rule_acc


# print((plot_df['실제'] == plot_df['prediction']).mean())
# print((plot_df['실제'] == plot_df['rule_label']).mean())

In [ ]:
front_df, tab_df.round(4)

In [ ]:
sns.barplot(
    data = front_df,
    x = 'Target',
    y = 'Score',
    hue = 'Predictor'
)


In [ ]:
front_df